In [50]:
import glob #to read files from folder
import os
import mne #used to analyze eef datasets
import numpy as np
import pandas
import matplotlib.pyplot as plt 

In [51]:
all_file_names=[file for file in glob.glob('dataverse_files/*.edf')] #importing files 
print(all_file_names)

['dataverse_files/s01.edf', 'dataverse_files/h01.edf', 'dataverse_files/h14.edf', 'dataverse_files/s14.edf', 'dataverse_files/s02.edf', 'dataverse_files/h02.edf', 'dataverse_files/h03.edf', 'dataverse_files/s03.edf', 'dataverse_files/s07.edf', 'dataverse_files/s13.edf', 'dataverse_files/h13.edf', 'dataverse_files/h07.edf', 'dataverse_files/h06.edf', 'dataverse_files/h12.edf', 'dataverse_files/s12.edf', 'dataverse_files/s06.edf', 'dataverse_files/s10.edf', 'dataverse_files/s04.edf', 'dataverse_files/h04.edf', 'dataverse_files/h10.edf', 'dataverse_files/h11.edf', 'dataverse_files/h05.edf', 'dataverse_files/s05.edf', 'dataverse_files/s11.edf', 'dataverse_files/s08.edf', 'dataverse_files/h08.edf', 'dataverse_files/h09.edf', 'dataverse_files/s09.edf']


In [52]:
print(all_file_path[0].split('/'))
healthy_file_path=[file for file in all_file_path if 'h' in file.split('/')[1]]
patient_file_path=[file for file in all_file_path if 's' in file.split('/')[1]]
print(len(healthy_file_path), len(patient_file_path))

['dataverse_files', 's01.edf']
14 14


Above: 28 EEG files (.edf) are loaded where h*.edf are healthy controls, and s*.edf are Parkinson's patients.

In [53]:
def read_data(file_path):
        data=mne.io.read_raw_edf(file_path, preload=True)
        data.set_eeg_reference()
        data.filter(l_freq=0.5,h_freq=45)
        epochs=mne.make_fixed_length_epochs(data, duration=5, overlap=1)
        array=epochs.get_data()
        return array

Standard pipeline of EEG recordings:
1. Average re-reference: subtract mean of electrodes (removes noise)
2. Bandpass filter (0.5-45 Hz): removes DC drifts and high-freq noise/powerline
3. Epoching: chops continuous recordings using 5s windows, 1s overlap

In [54]:
sample_data=read_data(healthy_file_path[0])

Extracting EDF parameters from dataverse_files/h01.edf...
Setting channel info structure...
Creating raw.info structure...
Reading 0 ... 231249  =      0.000 ...   924.996 secs...
EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 0.5 - 45 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 0.50
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 0.25 Hz)
- Upper passband edge: 45.00 Hz
- Upper transition bandwidth: 11.25 Hz (-6 dB cutoff frequency: 50.62 Hz)
- Filter length: 1651 samples (6.604 s)

Not setting metadata
231 matching events found
No baseline correction applied
0 projection items activated
Using data from preloaded Raw for 231 

In [55]:
sample_data.shape 

(231, 19, 1250)

In [56]:
%%capture
control_epochs_array=[read_data(data) for data in healthy_file_path]
patient_epochs_array=[read_data(data) for data in patient_file_path]

In [62]:
control_epochs_array[0].shape, control_epochs_array[1].shape

((231, 19, 1250), (216, 19, 1250))

In [61]:
control_epoch_labels=[ len(i)*[0] for i in control_epochs_array]
patient_epoch_labels=[ len(i)*[1] for i in patient_epochs_array]
len(control_epoch_labels), len(patient_epoch_labels)

(14, 14)

In [66]:
data_list=control_epochs_array + patient_epochs_array
label_list=control_epoch_labels + patient_epoch_labels

In [71]:
group_list=[[i]*len(j) for i, j in enumerate(data_list)]

28

In [74]:
data_array=np.vstack(data_list) #final data shape
label_array=np.hstack(label_list)
group_array=np.hstack(group_list)
print(data_array.shape, label_array.shape, group_array.shape)

(7201, 19, 1250) (7201,) (7201,)
